# DSE 200 — Day 4 Lab, part 4
## Models In Production

**Sections 11.1–11.4 match the slides · the DOJO is 11.4 · agent on: explain every cell.**

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while we talk. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost.

In [ ]:
!uv pip install --system -q mlflow

In [ ]:
# ▶ SETUP — run first. Rebuilds the Gradient Boosting notebook's data and model (about a minute).
import os, time, textwrap, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        print("downloaded", name, "from the course repo")
        return name
    except Exception as e:
        raise FileNotFoundError(f"{name} not found -- drag it into the Files pane (folder icon, left)") from e

df = pd.read_csv(find("minute_weather.csv.gz"), parse_dates=["hpwren_timestamp"])
df = df.drop_duplicates(subset=df.columns.drop("rowID"))            # the stuck logger (8.3)
df = df.sort_values("hpwren_timestamp").reset_index(drop=True)
df["dry_now"] = (df["relative_humidity"] < 25).astype(int)
df["dry_3h"]  = (df["relative_humidity"].shift(-180) < 25).astype(int)   # the label (8.4)
df = df.iloc[:-180].copy()

# features that look only backward (8.7)
df["hour"]  = df["hpwren_timestamp"].dt.hour
df["month"] = df["hpwren_timestamp"].dt.month
df["rh_change_1h"] = df["relative_humidity"] - df["relative_humidity"].shift(60)
df["rh_change_3h"] = df["relative_humidity"] - df["relative_humidity"].shift(180)
df["temp_change_1h"] = df["air_temp"] - df["air_temp"].shift(60)
df["pressure_change_3h"] = df["air_pressure"] - df["air_pressure"].shift(180)
sensors = ["air_pressure", "air_temp", "avg_wind_direction", "avg_wind_speed",
           "max_wind_direction", "max_wind_speed", "min_wind_direction",
           "min_wind_speed", "rain_accumulation", "rain_duration", "relative_humidity"]
features = sensors + ["hour", "month", "rh_change_1h", "rh_change_3h",
                      "temp_change_1h", "pressure_change_3h"]

# year 1 trains, year 2 chooses settings, year 3 is the final test (8.5)
year1 = df[df["hpwren_timestamp"] < "2012-09-10"]
year2 = df[(df["hpwren_timestamp"] >= "2012-09-10") & (df["hpwren_timestamp"] < "2013-09-10")]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]
model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
model.fit(year1[features], year1["dry_3h"],
          eval_set=[(year2[features], year2["dry_3h"])], verbose=False)

_READ = {
"11.1": ("6,825 of the 24,976 gap minutes get a different answer.",
         "Same model, same rows. Only the fill changed: the training mean (47.2%) vs the last hour's mean.\n"
         "Whichever one you validated, production runs the other. The model is fine; the pipeline is not.\n"
         "Save the fill rule WITH the model, and run the exact same code in both places."),
"11.3": ("Nothing. No error, no warning. The predictions just get worse.",
         "After the new sensor, accuracy falls from 0.918 to 0.867 while the model keeps answering confidently.\n"
         "Drift does not raise an exception. Only a check you scheduled can see it."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print(f"{len(df):,} minutes. Model trained; check_read() ready.")

# What Actually Breaks

*Slides: Models In Production · section 11.1*

Three things, in order of how often: **you cannot reproduce it**, **training and serving disagree**, **the world moves**.

## 11.1 Training And Serving Disagree

In the notebook you fill a missing value with the **training** mean. In production someone fills it with the last hour's mean, computed fresh on live data. Different number, different predictions.

In [ ]:
# 🔍 READ 11.1 -- the humidity sensor drops out for 5% of year-3 minutes.
#
#     Fill the gaps with year 1's mean humidity, or with the last hour's live mean. Same model.
#     Of the gap minutes, how many get a DIFFERENT prediction? None, a few, or many? Commit.

prediction = "???"
check_read("11.1", prediction)

In [ ]:
# ▶ DEMO
live = year3.copy()
gap = np.random.default_rng(0).random(len(live)) < 0.05          # 5% of minutes: no humidity reading
train_mean = year1["relative_humidity"].mean()                               # what the notebook used
last_hour = live["relative_humidity"].shift(1).rolling(60, min_periods=1).mean()   # what production used

filled_train = live[features].copy(); filled_train.loc[gap, "relative_humidity"] = train_mean
filled_live  = live[features].copy(); filled_live.loc[gap, "relative_humidity"] = last_hour[gap]
changed = model.predict(filled_train) != model.predict(filled_live)
print(f"{changed.sum():,} of {gap.sum():,} gap minutes get a different answer")

# Tracking Experiments

*Slides: Models In Production · section 11.2*

## 11.2 MLflow, Minimally

Record every run, automatically: the **parameters** you set, the **metrics** you got, the **model** itself. Four lines around code you already wrote. Then *"which settings gave us 0.91?"* stops being archaeology and becomes a query.

In [ ]:
# ▶ DEMO -- one tracked run
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")     # every run is a row in this small database file
mlflow.set_experiment("dry_3h")

with mlflow.start_run():
    mlflow.log_params({"learning_rate": 0.05, "max_depth": 3})
    m = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
    m.fit(year1[features], year1["dry_3h"], eval_set=[(year2[features], year2["dry_3h"])], verbose=False)
    mlflow.log_metric("year2_accuracy", m.score(year2[features], year2["dry_3h"]))
    mlflow.xgboost.log_model(m, name="model")

In [ ]:
# ▶ DEMO -- every run becomes a row you can sort
mlflow.search_runs(order_by=["metrics.year2_accuracy DESC"])[
    ["run_id", "params.learning_rate", "params.max_depth", "metrics.year2_accuracy"]]

On your own machine, `mlflow ui --backend-store-uri sqlite:///mlflow.db` in a terminal opens the same table in a browser. In Colab, `search_runs` is the UI.

**Versioning is three things, not one:** code (git), data (which snapshot, a path that is never overwritten), model (the fitted artefact, with the other two attached). When something goes wrong the question is always *"what exactly produced this prediction?"*

# The World Moves

*Slides: Models In Production · section 11.3*

## 11.3 Drift

The barometer is replaced in March 2014. The new one reports **inches of mercury** (about 27) instead of hPa (about 917). Nothing in the pipeline checks units. The model keeps answering, confidently, in the old world.

In [ ]:
# 🔍 READ 11.3 -- after the swap, what does the model do? Raise an error, warn, or something else? Commit.

prediction = "???"
check_read("11.3", prediction)

In [ ]:
# ▶ DEMO -- the same model on year 3, before and after a sensor swap
new_sensor = year3[features].copy()
swapped = year3["hpwren_timestamp"] >= "2014-03-01"
for col in ["air_pressure", "pressure_change_3h"]:
    new_sensor.loc[swapped, col] *= 0.02953                       # hPa -> inches of mercury

month = year3["hpwren_timestamp"].dt.to_period("M")
monthly = pd.DataFrame({
    "as trained": model.predict(year3[features]) == year3["dry_3h"],
    "new sensor": model.predict(new_sensor) == year3["dry_3h"],
}).groupby(month.values).mean()
monthly.plot(marker="o", figsize=(9, 3.5), title="accuracy by month, year 3"); plt.show()
monthly.round(3).tail(7)

# Evals Are The New Unit Test

*Slides: Models In Production · section 11.4 · the DOJO*

`assert mean([1, 2, 3]) == 2` passes or fails forever. A model's output is a distribution, not a value. **You assert on behaviour, not equality:**

1. the score on a **fixed** held-out set does not drop below a bar
2. predictions on known rows stay stable within a tolerance
3. the input still looks like what we trained on: the drift alarm, the one people skip

## 11.4 The DOJO

1. Wrap this morning's fit in an MLflow run (11.2 is the template). Log params, a metric and the model.
2. Change **one** knob, run again, and sort the two runs.
3. Write the one assertion you would put on a schedule.

In [ ]:
# ✍️ YOU TRY 11.4 (1) and (2) -- two tracked runs that differ in ONE setting
# TODO: copy the 11.2 cell, change one knob, run it

In [ ]:
# ✍️ YOU TRY 11.4 (3) -- a drift check you could run every month.
# Return True when this month's mean air_pressure is further from the training mean
# than 3 standard deviations of year 1's MONTHLY means.
train_monthly = year1.groupby(year1["hpwren_timestamp"].dt.month)["air_pressure"].mean()

def pressure_drifted(this_month):
    """this_month: a DataFrame of one month of live minutes."""
    return False    # TODO

In [ ]:
# ✅ CHECKPOINT 11.4 -- your check, run on every month of year 3, with and without the new sensor
live_new = year3.copy(); live_new.loc[swapped, "air_pressure"] *= 0.02953
for m_ in sorted(month.unique()):
    rows = (month == m_).values
    print(f"{m_}   as trained: {'ALARM' if pressure_drifted(year3[rows]) else 'ok':5}"
          f"   new sensor: {'ALARM' if pressure_drifted(live_new[rows]) else 'ok'}")
assert not any(pressure_drifted(year3[(month == m_).values]) for m_ in month.unique()), "false alarm on the real data"
assert pressure_drifted(live_new[(month == pd.Period("2014-04", "M")).values]), "missed the new sensor"
print("checkpoint passed")

### ✍️ And the question to answer, in a sentence

**If this model started getting worse, how would you find out?**
If the honest answer is "a customer would tell us", say that. It is the most common answer in industry.

*Your answer:*